# LSTM 时间序列预测

## 本节目标

LSTM 能更稳定地建模序列依赖。本节使用滑动窗口预测正弦序列的下一步。

完成本节后，你应该能够：

- 说清本节主题解决了什么问题。
- 读懂核心 PyTorch API 的输入输出。
- 运行最小代码示例并解释结果。
- 修改实验参数并观察变化。

## 背景问题

本节关注的问题是：LSTM 能更稳定地建模序列依赖。本节使用滑动窗口预测正弦序列的下一步。

学习时建议先运行代码，再回头看解释。对初学者来说，能观察 shape、loss、accuracy 或可视化结果，比只记概念更重要。

## 核心概念

- 输入和输出的 shape 必须明确。
- loss 用来描述模型当前预测和目标之间的差距。
- 优化器根据梯度更新模型参数。
- 实验观察需要结合曲线、数值和样本可视化。
- 调试时优先检查 shape、dtype、学习率和训练/评估模式。

这里的关键不是堆公式，而是把概念落实到可运行代码。

## 最小代码示例：构造窗口数据

这个代码块用于观察 `构造窗口数据`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

steps = np.linspace(0, 18 * np.pi, 460)
series = (np.sin(steps) + 0.08 * np.random.randn(len(steps))).astype(np.float32)
window = 20
xs, ys = [], []
for i in range(len(series) - window):
    xs.append(series[i:i+window])
    ys.append(series[i+window])
x_all = torch.tensor(np.array(xs)).unsqueeze(-1)
y_all = torch.tensor(np.array(ys)).unsqueeze(-1)
split = int(len(x_all) * 0.8)
x_train, y_train = x_all[:split], y_all[:split]
x_test, y_test = x_all[split:], y_all[split:]
print(x_train.shape, y_train.shape)

## 最小代码示例：训练 LSTM

这个代码块用于观察 `训练 LSTM`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, batch_first=True)
        self.head = nn.Linear(32, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

model = LSTMRegressor()
loss_fn = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []
for _ in range(100):
    pred = model(x_train)
    loss = loss_fn(pred, y_train)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

with torch.no_grad():
    test_pred = model(x_test)
print("final loss:", losses[-1])

## 完整实验：预测曲线

这个代码块用于观察 `预测曲线`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(y_test.numpy()[:140], label="target")
plt.plot(test_pred.numpy()[:140], label="prediction")
plt.legend()
plt.title("LSTM one-step prediction")
plt.grid(alpha=0.3)
plt.show()

## 实验观察

预测曲线通常能跟随整体趋势，但不会完全贴合噪声。窗口长度、隐藏维度和学习率都会影响结果。

从运行结果可以看到，代码输出不只是“能跑”，还可以帮助判断模型是否在按预期学习。建议把观察写进实验记录，而不是只保留最终准确率。

## 常见错误

- shape 不匹配：先打印输入、输出和标签 shape。
- dtype 不正确：分类标签通常是 `torch.long`，回归目标通常是 float。
- 忘记 `optimizer.zero_grad()`：梯度会累积。
- 学习率不合理：过大震荡，过小收敛慢。
- 评估阶段忘记 `model.eval()` 或 `torch.no_grad()`。

调试时可以先缩小数据集，确认模型能否在小样本上过拟合。

## 概念问答

**Q：为什么要从最小代码示例开始？**  
A：最小示例能隔离核心概念，降低同时排查多个问题的难度。

**Q：为什么每个实验都要看曲线或可视化？**  
A：单个最终数值不一定能解释训练过程，曲线和样本结果更容易暴露问题。

**Q：如果代码能运行但结果不好，先查什么？**  
A：优先检查数据、shape、label dtype、loss 使用方式和学习率。

## 延伸练习

- 尝试 window=5、40。
- 增加 GRU 对比实验。

这些练习不需要一次完成。建议每次只改一个变量，并记录改动前后的结果。

## 小结

本节把一个深度学习概念落到了可运行的 PyTorch 实验中。继续学习下一节前，建议确认自己能解释：输入是什么、模型做了什么、loss 如何计算、结果是否符合预期。